# 02 · HAR-RV benchmarks (HAR-RV and log-HAR)

One regression per stock (Corsi, 2009):

$$RV_{t+1} = \beta_0 + \beta_D RV_t + \beta_W RV^{(5)}_t + \beta_M RV^{(22)}_t + \varepsilon_{t+1}$$

* **HAR-RV** – the classic model in levels, by OLS.
* **log-HAR** – the same regression in logs; forecast $= \exp(\hat y + s^2/2)$. Logs damp the influence of a
  few extreme days on the coefficients. It is the stronger econometric benchmark and the fair
  comparison for the LSTMs, which also see log inputs.

Validation forecasts use coefficients estimated on 2016–2022. Test forecasts re-estimate on
2016–2024 (the LSTMs are *not* re-trained, so this gives HAR a small advantage).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from volforecast import config as C, data_pipeline as dp, har
from volforecast.evaluation import metric_table

panel = pd.read_parquet(C.PROCESSED_DIR / "daily_panel.parquet")
universe = pd.read_csv(C.PROCESSED_DIR / "universe.csv")["Symbol"].tolist()
data = panel[panel["Symbol"].isin(universe)].dropna(subset=["RV_5", "RV_22", "RV_t1"])
train, valid, test = dp.split_by_target_date(data)
print(len(universe), "stocks | train", len(train), "| valid", len(valid), "| test", len(test))

95 stocks | train 152935 | valid 46599 | test 29774


In [2]:
results, coefs = {}, {}
for name, use_log in [("HAR-RV", False), ("log-HAR", True)]:
    fv, cv = har.har_forecasts(train, valid, log=use_log, name=name)
    ft, ct = har.har_forecasts(pd.concat([train, valid]), test, log=use_log, name=name)
    results[name] = (fv, ft)
    coefs[name] = ct
    print(f"{name}: negative forecasts floored -> valid {fv[name + '_was_negative'].sum()}, "
          f"test {ft[name + '_was_negative'].sum()}")

for split, k in [("valid", 0), ("test", 1)]:
    f = results["HAR-RV"][k].merge(results["log-HAR"][k][["Symbol", "Date", "log-HAR"]],
                                   on=["Symbol", "Date"])
    f.drop(columns=[c for c in f if c.endswith("_was_negative")]).to_csv(
        C.FORECAST_DIR / f"har_{split}_forecasts.csv", index=False)
    print(f"\n{split.upper()}")
    display(metric_table(f, ["HAR-RV", "log-HAR"]))

HAR-RV: negative forecasts floored -> valid 0, test 0


log-HAR: negative forecasts floored -> valid 0, test 0



VALID


,Obs,MSE,MAE,QLIKE
Model,,,,
HAR-RV,46599,5.932688e-07,0.000223,0.260585
log-HAR,46599,6.015800e-07,0.000197,0.229848



TEST


,Obs,MSE,MAE,QLIKE
Model,,,,
HAR-RV,29774,9.302536e-08,0.000166,0.228602
log-HAR,29774,8.597350e-08,0.000143,0.196545


## Average HAR coefficients (test-period fit, 2016–2024)
In levels the daily coefficient is pulled around by a few extreme days; in logs the
daily–weekly–monthly cascade is stable and sums close to one (high persistence).

In [3]:
summary = pd.concat({k: v[["const", "beta_D", "beta_W", "beta_M"]].describe().loc[["mean", "50%", "std"]]
                     for k, v in coefs.items()})
for k, v in coefs.items():
    v.to_csv(C.TABLE_DIR / f"coefficients_{k}.csv", index=False)
summary.round(3)

const  beta_D  beta_W  beta_M
HAR-RV  mean  0.000   0.229   0.365   0.125
        50%   0.000   0.231   0.360   0.123
        std   0.000   0.121   0.171   0.103
log-HAR mean -1.660   0.347   0.246   0.212
        50%  -1.586   0.347   0.252   0.214
        std   0.422   0.054   0.058   0.060